In [1]:
%store -r selected_train_features

%store -r resampled_train_labels

%store -r selected_test_features

%store -r encoded_test_labels

%store -r encoded_valid_labels

%store -r selected_valid_features

In [2]:
selected_train_features_final = selected_train_features

selected_test_features_final = selected_test_features

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

In [4]:
models = {
    "Random Forest": RandomForestClassifier(
    n_estimators=500,
    criterion="gini",
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    random_state=42,
    n_jobs=-1
),
    "Decision Tree":  DecisionTreeClassifier(
    criterion="gini",
    splitter="best",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    class_weight=None,
    random_state=42
),
    "Logistic Regression": LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=500,
    class_weight=None,
    random_state=42
),
    "KNN": KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="euclidean",
    p=2,
    n_jobs=-1
),
    "XGBoost": XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=8,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=7,
    eval_metric="mlogloss",
    reg_alpha=0,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1
)
}

In [5]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [6]:
# Train all classification models
for model_name, model in models.items():
    print(f"Training {model_name}...")

    model.fit(
        selected_train_features_final,
        resampled_train_labels
    )

print("\nAll models trained successfully.")

Training Random Forest...
Training Decision Tree...
Training Logistic Regression...


c:\Users\Meiappan\Music\Projects\Forest Cover Classification\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training KNN...
Training XGBoost...

All models trained successfully.


In [7]:
# Compare model accuracy on the training set
for model_name, model in models.items():
    train_predictions = model.predict(
        selected_train_features_final
    )

    train_accuracy = accuracy_score(
        resampled_train_labels,
        train_predictions
    )

    print(f"{model_name}: {train_accuracy:.4f}")

Random Forest: 0.9999
Decision Tree: 1.0000
Logistic Regression: 0.7503
KNN: 1.0000
XGBoost: 0.9979


In [8]:
# Display classification reports for all models on the training set
for model_name, model in models.items():
    train_predictions = model.predict(selected_train_features_final)

    print("\n" + "=" * 70)
    print(f"{model_name} - TRAINING CLASSIFICATION REPORT")
    print("=" * 70)

    print(
        classification_report(
            resampled_train_labels,
            train_predictions,
            digits=2
        )
    )


Random Forest - TRAINING CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     72150
           1       1.00      1.00      1.00     72150
           2       1.00      1.00      1.00     72150
           3       1.00      1.00      1.00     72150
           4       1.00      1.00      1.00     72150
           5       1.00      1.00      1.00     72150
           6       1.00      1.00      1.00     72150

    accuracy                           1.00    505050
   macro avg       1.00      1.00      1.00    505050
weighted avg       1.00      1.00      1.00    505050


Decision Tree - TRAINING CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     72150
           1       1.00      1.00      1.00     72150
           2       1.00      1.00      1.00     72150
           3       1.00      1.00      1.00     72150
           4       1.00      1.00   

In [9]:
# Display confusion matrices for all models on the training set
for model_name, model in models.items():
    train_predictions = model.predict(selected_train_features_final)

    print("\n" + "=" * 70)
    print(f"{model_name} - TRAINING CONFUSION MATRIX")
    print("=" * 70)

    print(
        confusion_matrix(
            resampled_train_labels,
            train_predictions
        )
    )


Random Forest - TRAINING CONFUSION MATRIX
[[72148     0     0     0     2     0     0]
 [    0 72150     0     0     0     0     0]
 [    0     0 72150     0     0     0     0]
 [    0     0     0 72150     0     0     0]
 [    3     0     0     0 72118     0    29]
 [    0     0     0     0     0 72150     0]
 [    0     0     0     0    18     0 72132]]

Decision Tree - TRAINING CONFUSION MATRIX
[[72150     0     0     0     0     0     0]
 [    0 72150     0     0     0     0     0]
 [    0     0 72150     0     0     0     0]
 [    0     0     0 72150     0     0     0]
 [    0     0     0     0 72150     0     0]
 [    0     0     0     0     0 72150     0]
 [    0     0     0     0     0     0 72150]]

Logistic Regression - TRAINING CONFUSION MATRIX
[[56890     0  2318     0  7684  1347  3911]
 [    0 64761  1885     0     0  5504     0]
 [ 2041  4616 50475     0     0 15018     0]
 [   76     0     0 69535     0     0  2539]
 [ 9637     2    63   150 47085    35 15178]
 [ 1923 

In [10]:
# Compare model accuracy on the validation set
for model_name, model in models.items():
    valid_predictions = model.predict(selected_valid_features)

    valid_accuracy = accuracy_score(
        encoded_valid_labels,
        valid_predictions
    )

    print(f"{model_name}: {valid_accuracy:.4f}")

Random Forest: 0.9430
Decision Tree: 0.9192
Logistic Regression: 0.6674
KNN: 0.7664
XGBoost: 0.9493


In [11]:
# Display classification reports for all models on the validation set
for model_name, model in models.items():
    valid_predictions = model.predict(selected_valid_features)

    print("\n" + "=" * 70)
    print(f"{model_name} - VALIDATION CLASSIFICATION REPORT")
    print("=" * 70)

    print(
        classification_report(
            encoded_valid_labels,
            valid_predictions,
            digits=2
        )
    )


Random Forest - VALIDATION CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       461
           1       0.90      0.96      0.93       324
           2       0.83      0.83      0.83       324
           3       0.93      0.96      0.95       324
           4       0.97      0.96      0.97     15460
           5       0.84      0.84      0.84       324
           6       0.88      0.91      0.90      4666

    accuracy                           0.94     21883
   macro avg       0.88      0.91      0.89     21883
weighted avg       0.94      0.94      0.94     21883


Decision Tree - VALIDATION CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.68      0.84      0.75       461
           1       0.90      0.91      0.90       324
           2       0.77      0.75      0.76       324
           3       0.91      0.94      0.92       324
           4       0.97      0.9

In [12]:
# Display confusion matrices for all models on the validation set
for model_name, model in models.items():
    valid_predictions = model.predict(selected_valid_features)

    print("\n" + "=" * 70)
    print(f"{model_name} - VALIDATION CONFUSION MATRIX")
    print("=" * 70)

    print(
        confusion_matrix(
            encoded_valid_labels,
            valid_predictions
        )
    )


Random Forest - VALIDATION CONFUSION MATRIX
[[  407     0     8     0    40     3     3]
 [    0   311     5     0     0     8     0]
 [    4    14   268     0     0    38     0]
 [    1     0     0   310     0     0    13]
 [   66     0    10     5 14811     4   564]
 [    3    19    28     0     1   273     0]
 [   17     0     2    17   374     0  4256]]

Decision Tree - VALIDATION CONFUSION MATRIX
[[  386     0     8     0    50     8     9]
 [    0   294     9     0     0    21     0]
 [    5    13   242     0     3    61     0]
 [    1     0     0   304     0     0    19]
 [  147     0     9     6 14421     6   871]
 [    7    19    43     0     5   250     0]
 [   19     0     3    25   400     1  4218]]

Logistic Regression - VALIDATION CONFUSION MATRIX
[[  329     0    27     0    73     8    24]
 [    0   276    10     0     0    38     0]
 [   13    23   220     0     0    68     0]
 [    2     0     0   303     0     0    19]
 [ 2061     0    14    29 10178     6  3172]
 [

In [13]:
# Compare model accuracy on the test set
for model_name, model in models.items():
    test_predictions = model.predict(selected_test_features_final)

    test_accuracy = accuracy_score(
        encoded_test_labels,
        test_predictions
    )

    print(f"{model_name}: {test_accuracy:.4f}")

Random Forest: 0.9448
Decision Tree: 0.9210
Logistic Regression: 0.6680
KNN: 0.7664
XGBoost: 0.9505


In [14]:
# Display classification reports for all models on the test set
for model_name, model in models.items():
    test_predictions = model.predict(selected_test_features_final)

    print("\n" + "=" * 70)
    print(f"{model_name} - TEST CLASSIFICATION REPORT")
    print("=" * 70)

    print(
        classification_report(
            encoded_test_labels,
            test_predictions,
            digits=2
        )
    )


Random Forest - TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.86      0.88      0.87       460
           1       0.92      0.96      0.94       324
           2       0.81      0.74      0.77       324
           3       0.92      0.98      0.94       324
           4       0.97      0.96      0.97     15461
           5       0.74      0.85      0.79       324
           6       0.89      0.91      0.90      4667

    accuracy                           0.94     21884
   macro avg       0.87      0.90      0.88     21884
weighted avg       0.95      0.94      0.95     21884


Decision Tree - TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.75      0.86      0.80       460
           1       0.92      0.90      0.91       324
           2       0.71      0.67      0.69       324
           3       0.89      0.97      0.93       324
           4       0.97      0.94      0.95 

In [15]:
# Display confusion matrices for all models on the test set
for model_name, model in models.items():
    test_predictions = model.predict(selected_test_features_final)

    print("\n" + "=" * 70)
    print(f"{model_name} - TEST CONFUSION MATRIX")
    print("=" * 70)

    print(
        confusion_matrix(
            encoded_test_labels,
            test_predictions
        )
    )


Random Forest - TEST CONFUSION MATRIX
[[  407     0     8     0    35     8     2]
 [    0   312     2     0     0    10     0]
 [    4    13   240     0     0    67     0]
 [    0     0     0   316     0     1     7]
 [   47     0    13     2 14891    10   498]
 [    2    15    30     0     0   277     0]
 [   13     0     3    27   390     0  4234]]

Decision Tree - TEST CONFUSION MATRIX
[[  396     0     9     0    42     8     5]
 [    0   292    11     0     0    21     0]
 [    9    10   216     0     2    85     2]
 [    1     0     0   315     0     0     8]
 [   99     0    10     4 14512    11   825]
 [    2    14    55     0     2   251     0]
 [   23     0     4    36   429     1  4174]]

Logistic Regression - TEST CONFUSION MATRIX
[[  334     0    28     0    53    13    32]
 [    0   274    10     0     0    40     0]
 [   17    31   200     0     0    76     0]
 [    2     0     0   305     0     0    17]
 [ 1996     0    19    31 10254     9  3152]
 [   14    34    89 

In [16]:
from sklearn.metrics import f1_score
import pandas as pd

# Store test performance for all models
test_results = {}

for model_name, model in models.items():
    test_predictions = model.predict(selected_test_features_final)

    test_results[model_name] = {
        "Accuracy": accuracy_score(
            encoded_test_labels,
            test_predictions
        ),
        "Macro F1": f1_score(
            encoded_test_labels,
            test_predictions,
            average="macro"
        ),
        "Weighted F1": f1_score(
            encoded_test_labels,
            test_predictions,
            average="weighted"
        )
    }

# Display test model comparison
test_results_dataframe = pd.DataFrame(
    test_results
).T.sort_values(
    by="Macro F1",
    ascending=False
)

print("=" * 70)
print("FINAL TEST MODEL COMPARISON")
print("=" * 70)

print(test_results_dataframe)

FINAL TEST MODEL COMPARISON
                     Accuracy  Macro F1  Weighted F1
XGBoost              0.950466  0.895253     0.950252
Random Forest        0.944846  0.884813     0.945079
Decision Tree        0.921038  0.836971     0.922186
KNN                  0.766359  0.658864     0.780594
Logistic Regression  0.667977  0.611465     0.702336


In [17]:
# Select the best model based on final test Macro F1
best_model_name = test_results_dataframe["Macro F1"].idxmax()

best_model = models[best_model_name]

print("\n" + "=" * 70)
print("BEST MODEL")
print("=" * 70)

print("Model:", best_model_name)
print(
    "Test Accuracy:",
    f"{test_results_dataframe.loc[best_model_name, 'Accuracy']:.4f}"
)
print(
    "Test Macro F1:",
    f"{test_results_dataframe.loc[best_model_name, 'Macro F1']:.4f}"
)
print(
    "Test Weighted F1:",
    f"{test_results_dataframe.loc[best_model_name, 'Weighted F1']:.4f}"
)


BEST MODEL
Model: XGBoost
Test Accuracy: 0.9505
Test Macro F1: 0.8953
Test Weighted F1: 0.9503


In [18]:
import joblib

# Save the best-performing model
joblib.dump(
    best_model,
    "best_model.pkl"
)

print("Best model saved successfully.")
print("Saved model:", best_model_name)

Best model saved successfully.
Saved model: XGBoost


In [19]:
%store selected_test_features_final

%store selected_train_features_final

Stored 'selected_test_features_final' (DataFrame)
Stored 'selected_train_features_final' (DataFrame)
